# 02. Road Backbone Analysis & Foundation

This notebook provides a detailed walkthrough of the **Backbone Foundation** creation process. We transform raw geospatial road data into a discrete point-based network enriched with traffic intensity, electrical capacity, and proximity to existing infrastructure.

## Objectives
1. **Discretize** LineString geometries from KMZ files into equidistant points (every 200m).
2. **Map Traffic** intensity from high-resolution road segments to our backbone points.
3. **Analyze Proximity** to existing EV charging stations and gas stations.
4. **Visualize Gaps** in the current network to identify optimal locations for new ultra-fast chargers.

In [ ]:
!git clone https://github.com/JJR9903/Iberdrola-Datathon.git

In [ ]:
cd "/content/Iberdrola-Datathon"

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
import os
import numpy as np
import tomllib
import sys
import polars as pl
import statsmodels.api as sm
import pmdarima as pm
from datetime import date
from dateutil.relativedelta import relativedelta



# Change directory to root
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Add scripts directory to path to allow module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from scripts.sync_cloud import sync_standardized_data
from scripts.processing.create_backbone_foundation import (
    discretize_backbone_roads,
    map_traffic_to_points,
    assign_nearest_charging_stations,
    assign_nearest_gas_stations,
    assign_grid_capacity
)


In [ ]:
# 1. Sync data (Ensures silver layer is present)
sync_standardized_data()


In [ ]:
# 2. Load Config
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
params = config['steps']['backbone_foundation']


In [ ]:
params['capacity_path']

In [ ]:
# 3. Load Standardized Data

print("\n🚀 Loading Standardized Data...")
gdf_roads = gpd.read_parquet(params['roads_path'])
gdf_traffic = gpd.read_parquet(params['traffic_path'])
gdf_chargers = gpd.read_parquet(params['chargers_path'])
gdf_gas = gpd.read_parquet(params['gas_stations_path'])
gdf_capacity = gpd.read_parquet(params['capacity_path'])
pl_registrations = pl.read_parquet('data/standardized/vehicle_registrations.parquet')

print(f" - Roads: {len(gdf_roads)} segments")
print(f" - Traffic: {len(gdf_traffic)} segments")
print(f" - Chargers: {len(gdf_chargers)} sites")
print(f" - Gas Stations: {len(gdf_gas)} sites")
print(f" - Electric Capacity: {len(gdf_capacity)} sites")
print(f" - Vehicle Registrations: {len(pl_registrations)}")

print("\n✅ Data loaded correctly.")

In [ ]:
os.getcwd()

visual inspection of the road dataset

In [ ]:
gdf_roads.head()

visual inspection of the traffic dataset

In [ ]:
gdf_traffic.head()

visual inspection of the chargers dataset

In [ ]:
gdf_chargers.head()

visual inspection of the gas stations dataset

In [ ]:
gdf_gas.head()

visual inspection of the vehicle registrations dataset

In [ ]:
pl_registrations.head()

# Step 1: EV registrations forecast

In [ ]:
pl_registrations = pl_registrations.filter(
   ( pl.col('date') >= date(2015, 1, 1) ) & (pl.col('date') < date(2026, 1, 1)) 
)

In [ ]:
non_ev_registrations = pl_registrations.filter(pl.col('propulsion') != 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')    
).sort("date").to_pandas().set_index('date')

ev_registrations = pl_registrations.filter(pl.col('propulsion') == 'Eléctrico').group_by(
    pl.col("date").dt.truncate("1mo")
).agg(
    pl.col('brand').count().alias('registrations')
).sort("date").to_pandas().set_index('date')


## step 1.2 EV forecast

In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_EV_model = pm.auto_arima(np.log(ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_EV = auto_EV_model.order
best_seasonal_order_EV = auto_EV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_EV} & seasonal_order={best_seasonal_order_EV}")

EV_model = sm.tsa.statespace.SARIMAX(
    np.log(ev_registrations['registrations']), 
    order=best_order_EV, 
    seasonal_order=best_seasonal_order_EV
)
EV_results = EV_model.fit(disp=False)

forecast_steps = 24
EV_model_forecast = EV_results.get_forecast(steps=forecast_steps)

EV_model_forecast = (EV_model_forecast.summary_frame(alpha=0.5))
EV_fc_mean = round(np.exp(EV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': EV_fc_mean
}, index=forecast_dates)


ev_registrations = pd.concat([ev_registrations, df_ev_forecast]).rename(columns={'registrations':'ev_registrations'})

ev_registrations.tail(12)

## 1.3 Non EV Registrations

In [ ]:
# Selección automática del modelo con Auto-ARIMA
auto_nEV_model = pm.auto_arima(np.log(non_ev_registrations['registrations']),
                      test='adf',        # ADF Test
                      m=12,              # Estacionalidad de 12 meses
                      seasonal=True,     # Modelo SARIMA
                      trace=False,
                      error_action='ignore',
                      suppress_warnings=True,
                      stepwise=True)


# Ajuste del modelo SARIMAX con los parámetros óptimos de auto_arima
best_order_nEV = auto_nEV_model.order
best_seasonal_order_nEV = auto_nEV_model.seasonal_order

print(f"using a SARIMAX with order={best_order_nEV} & seasonal_order={best_seasonal_order_nEV} with drift")

nEV_model = sm.tsa.statespace.SARIMAX(
    np.log(non_ev_registrations['registrations']), 
    trend='c' if auto_nEV_model.with_intercept==True else 'n',
    order=best_order_nEV, 
    seasonal_order=best_seasonal_order_nEV
)
nEV_results = nEV_model.fit(disp=False)

forecast_steps = 24
nEV_model_forecast = nEV_results.get_forecast(steps=forecast_steps)

nEV_model_forecast = (nEV_model_forecast.summary_frame(alpha=0.5))
nEV_fc_mean = round(np.exp(nEV_model_forecast['mean']))

# Crear DataFrame para el pronóstico
last_date = non_ev_registrations.index.max()
forecast_dates = [last_date + relativedelta(months=i) for i in range(1, forecast_steps + 1)]

df_ev_forecast = pd.DataFrame({
    'registrations': nEV_fc_mean
}, index=forecast_dates)


non_ev_registrations = pd.concat([non_ev_registrations, df_ev_forecast]).rename(columns={'registrations':'non_ev_registrations'})

non_ev_registrations

In [ ]:
vehicle_registrations = ev_registrations.merge(non_ev_registrations,how='inner',left_index=True, right_index=True)
vehicle_registrations

In [ ]:
vehicle_registrations_total = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations["non_ev_registrations"].sum()],
})

vehicle_registrations_total["pct_ev_registrations"] = (
    vehicle_registrations_total["total_ev_registrations"] /
    (vehicle_registrations_total["total_ev_registrations"] + vehicle_registrations_total["total_non_ev_registrations"])
)
vehicle_registrations_total

In [ ]:
vehicle_registrations_2027 = pd.DataFrame({
    "total_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["ev_registrations"].sum()],
    "total_non_ev_registrations": [vehicle_registrations.loc[vehicle_registrations.index>='2027-01-01']["non_ev_registrations"].sum()],
})

vehicle_registrations_2027["pct_ev_registrations"] = (
    vehicle_registrations_2027["total_ev_registrations"] /
    (vehicle_registrations_2027["total_ev_registrations"] + vehicle_registrations_2027["total_non_ev_registrations"])
)
vehicle_registrations_2027

## Step 1: Discretization
The raw road backbone is provided as high-level `LineString` geometries in a KMZ file. To perform precise point-based analysis (like calculating the distance to the exact nearest charger), we convert these lines into points sampled every 200 meters.

**Assumptions:**
*   A **200m interval** provides a good balance between spatial resolution and computational efficiency.
*   We preserve the original `backbone_id` to maintain traceability to the source data.

In [ ]:
params['sampling_interval_m']

In [ ]:
gdf_points = discretize_backbone_roads(
    gdf_roads,
    sampling_interval_m=params['sampling_interval_m']
)
print(f"Generated {len(gdf_points)} points along the backbone.")
gdf_points.head()

### Visualization: Base Point Network
Below we see the result of the discretization. Each red dot represents a sampling point where we will later attach traffic and infrastructure data.

In [ ]:
m0 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
gdf_plot = gdf_points.to_crs(4326)

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='red',
        fill=True
    ).add_to(m0)
m0

## Step 2: Traffic Intensity Mapping
Traffic data is usually collected on specific road segments. We map this data to our backbone points using a spatial buffer.

**Assumptions & Logic:**
*   **Buffer Radius (50m)**: We look for road segments within 50m of each point.
*   **Neighbor Validation**: To avoid "cross-talk" from overpasses or intersecting roads, we only accept a segment if it intersects at least two consecutive points on the same backbone. This ensures the segment actually runs longitudinal to the backbone.

In [ ]:
params['traffic_columns']

In [ ]:
params['buffer_radius_m']

In [ ]:
params['traffic_columns']

In [ ]:
gdf_points = map_traffic_to_points(
    gdf_points,
    gdf_traffic,
    params['traffic_columns'],
    params['buffer_radius_m']
)
gdf_points[params['traffic_columns']].describe()


### Map 1: Road & Traffic
Points are colored based on their traffic intensity (`total_max`). Green indicates light traffic, yellow mid traffic, and red heavy traffic.

In [ ]:
import branca.colormap as cm

m1 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=gdf_points['total_max'].quantile(0.95))
colormap.caption = 'Traffic Intensity (Total Max)'
colormap.add_to(m1)

gdf_plot = gdf_points.to_crs(4326)
for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=colormap(row['total_max']),
        fill=True,
        tooltip=f"Traffic: {row['total_max']}"
    ).add_to(m1)
m1

## Step 3: Infrastructure Proximity
We now calculate the distance to the nearest existing EV charging station and gas station.

**Assumptions:**
*   The nearest charger is identified using a spatial `nearest` join.
*   Distances are calculated in meters using the metric CRS (EPSG:3042).

In [ ]:
params['max_distance_proximity']

In [ ]:
gdf_points = assign_nearest_charging_stations(
    gdf_points = gdf_points,
    gdf_chargers = gdf_chargers,
    max_distance = params['max_distance_proximity']
)

In [ ]:
gdf_points = assign_nearest_gas_stations(
    gdf_points = gdf_points,
    gdf_gas = gdf_gas,
    max_distance = params['max_distance_proximity']
)
print("Proximity analysis completed.")

### Map 2: Road & Current EV Chargers
This map highlights the areas with existing infrastructure. Points are colored by distance to the nearest charger (Green = Close, Red = Far).

In [ ]:
gdf_plot = gdf_points.to_crs(4326)

import branca.colormap as cm

m2 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')
dist_colormap = cm.LinearColormap(['green', 'yellow', 'red'], vmin=0, vmax=max(gdf_plot['dist_charger_m']))
dist_colormap.caption = 'Distance to Nearest Charger (m)'
dist_colormap.add_to(m2)

# Explicitly fill NaN values in 'dist_charger_m' before iteration
gdf_plot['dist_charger_m'] = gdf_plot['dist_charger_m'].fillna(max(gdf_plot['dist_charger_m']))

for _, row in gdf_plot.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color=dist_colormap(min(row['dist_charger_m'], 100000)), # Use the cleaned column directly
        fill=True
    ).add_to(m2)
m2

# Step 4: Optimization

In [ ]:
from scripts.processing.optimize_grid_aware_placement import generate_smart_candidates, solve_grid_aware_optimization, report

In [ ]:
ev_traffic_pct = vehicle_registrations_2027['pct_ev_registrations'].iloc[0]
print(ev_traffic_pct)

In [ ]:
need_charge_pct = gdf_points['medio_max'].sum() / gdf_points['total_max'].sum()
need_charge_pct

In [ ]:
max_chargers_per_site =  gdf_chargers['charger_count'].max()
max_chargers_per_site

In [ ]:
coverage_threshold_m = 30000
sessions_per_charger = 24
substation_threshold_m = 10000
power_per_charger_kw = 150
penalty_coverage = 1e6
penalty_supply = 1e4
penalty_grid_upgrade = 1e2

In [ ]:
df_cand = generate_smart_candidates(
    gdf_backbone = gdf_points, 
    gdf_chargers = gdf_chargers, 
    gdf_gas = gdf_gas, 
    ev_traffic_pct = ev_traffic_pct, 
    need_charge_pct = need_charge_pct, 
    coverage_threshold_m = coverage_threshold_m, 
    max_chargers_per_site = max_chargers_per_site, 
    sessions_per_charger = sessions_per_charger
    )

In [ ]:
df_res, grid_slacks = solve_grid_aware_optimization(
        gdf_backbone = gdf_points, 
        df_cand = df_cand, 
        gdf_grid = gdf_capacity,
        ev_traffic_pct = ev_traffic_pct,
        need_charge_pct= need_charge_pct,
        coverage_threshold_m = coverage_threshold_m,
        substation_threshold_m = substation_threshold_m,
        max_chargers_per_site = max_chargers_per_site,
        sessions_per_charger = sessions_per_charger,
        power_per_charger_kw = power_per_charger_kw,
        penalty_coverage = penalty_coverage,
        penalty_supply = penalty_supply,
        penalty_grid_upgrade = penalty_grid_upgrade
    )



In [ ]:
report(df_res, grid_slacks)

In [ ]:
gdf_capacity.loc[gdf_capacity['capacity_kw']>0]

In [ ]:
df_res

In [ ]:
gdf_res = gpd.GeoDataFrame(df_res, geometry='geometry', crs=gdf_backbone.crs)
gdf_res.to_parquet("data/processed/grid_aware_optimized_sites.parquet")

In [ ]:
gdf_res

## Output construction

Add capacity needed

In [ ]:
df_res['estimated_demand_kw'] = df_res['final_n'] * 150

Add Longitude and Latitude

In [ ]:
df_res['latitude'] = df_res.geometry.y
df_res['longitude'] = df_res.geometry.x

Filtered only points with a new charging station

In [ ]:
df_stations = df_res[(df_res['is_open']==1)&(df_res['type']!='existing')]

Add Loc Id for every station

In [ ]:
df_stations['location_id'] = [f"IBE_{i:02d}" for i in range(1, len(df_stations) + 1)]

Add road name

In [ ]:
df_stations_indexed = df_stations.set_index('loc_ID')

df_stations_with_road = gpd.sjoin_nearest(
    df_stations_indexed,
    gdf_points[['road_name', 'geometry']],
    how='left', 
)

df_stations_with_road = df_stations_with_road.loc[~df_stations_with_road.index.duplicated(keep='first')]

df_stations_with_road.rename(columns={'road_name': 'nearest_road_name'}, inplace=True)

df_stations = df_stations.set_index('loc_ID')
df_stations['route_segment'] = df_stations_with_road['nearest_road_name']
df_stations = df_stations.reset_index() 
